In [ ]:
import concurrent
import requests
from bs4 import BeautifulSoup
import json
import time
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from collections import deque
import logging

#Use the logging library instead of print() for proper error/notification management when deploying 
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define the base URL and starting URL
BASE_URL = "https://mifosforge.jira.com"
START_URL = "https://mifosforge.jira.com/wiki/spaces/projects/overview"

def get_page_content(url):
    """Fetch the content of the page at the specified URL."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        logging.error(f"Request failed for {url}: {e}")
        return None

def extract_links(html, base_url):
    """Extract and return all valid links from the given HTML content."""
    soup = BeautifulSoup(html, 'html.parser')
    links = set()
    for a in soup.find_all('a', href=True):
        href = a['href']
        full_url = urljoin(base_url, href)
        if full_url.startswith(base_url):
            links.add(full_url)
    return list(links)

def extract_data(html):
    """Extract visible text and hidden content from the HTML."""
    soup = BeautifulSoup(html, 'html.parser')
    text = soup.get_text(separator='\n', strip=True)
    hidden_content = [element.get_text(strip=True) for element in soup.find_all(style=True) if "display:none" in element.get("style", "")]
    return {"text": text, "hidden": hidden_content}

def scrape_site(start_url, max_urls=5, max_workers=5):
    """Scrape the site starting from the start_url up to max_urls pages using max_workers threads."""
    visited = set([start_url])
    data = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor: #create threadppol with max_workers threads
        # Dictionary mapping: Future object -> URL which is being fetched
        future_to_url = {executor.submit(get_page_content, start_url): start_url} # Submit first task to fetch the starting URL

        # while there are still tasks running AND we haven't reached the max URL count
        while future_to_url and len(data) < max_urls:
            # Wait for at least one future to complete, then process the completed ones
            done, not_done = concurrent.futures.wait(
                future_to_url.keys(), 
                return_when=concurrent.futures.FIRST_COMPLETED
            )

            for future in done:
                url = future_to_url[future]
                html = future.result()

                if html:
                    #save data
                    data[url] = extract_data(html)
                    logging.info(f"Successfully scraped: {url} ({len(data)}/{max_urls})")

                    #if we have already scraped enough URLs, break out of the loop
                    if len(data) >= max_urls:
                        break

                    #extract new links and submit new tasks for them
                    new_links = extract_links(html, BASE_URL)
                    for link in new_links:
                        if link not in visited:
                            visited.add(link)
                            future_to_url[executor.submit(get_page_content, link)] = link

            #delay avoid ddosing the server
            time.sleep(0.5) 

    return data

if __name__ == "__main__":
    max_urls = 3
    max_workers = 5
    scraped_data = scrape_site(START_URL, max_urls, max_workers)

    with open("scraped_data.json", "w", encoding="utf-8") as f:
        json.dump(scraped_data, f, ensure_ascii=False, indent=4)

    logging.info("Custom Scraping completed and data saved to scraped_data.json")


2026-03-18 08:54:00,536 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projects/overview (1/3)
2026-03-18 08:54:01,167 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projects/overview (1/3)
2026-03-18 08:54:01,812 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projects/overview (1/3)
2026-03-18 08:54:02,445 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projects/pages/6816200/Archived+Volunteer+Projects (2/3)
2026-03-18 08:54:02,578 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projects/overview (2/3)
2026-03-18 08:54:03,198 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projects/pages/6816200/Archived+Volunteer+Projects (2/3)
2026-03-18 08:54:03,314 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projects/overview (2/3)
2026-03-18 08:54:03,495 - INFO - Successfully scraped: https://mifosforge.jira.com/wiki/spaces/projec

In [ ]:
# Install with pip install firecrawl-py
# Install dotenv with pip install python-dotenv for environment variable management
import os
import json
import logging
from firecrawl import FirecrawlApp
from dotenv import load_dotenv

#Set up logging instead of print()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

#Use python-dotenv for API Key security
# Load environment variables from file .env. Never hardcode API key in source code.
load_dotenv()
FIRECRAWL_API_KEY = os.getenv('FIRECRAWL_API_KEY')

if not FIRECRAWL_API_KEY:
  logging.error("FIRECRAWL_API_KEY is not set. Please set it in your environment or .env file.")
else:
  app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)  

  logging.info("Starting Firecrawl extraction...")
  crawl_result = app.crawl_url('https://mifosforge.jira.com/wiki/spaces/projects/overview', params={
  'limit': 2, #add limit accordingly to crawl for sublinks
  'scrapeOptions': {
    'formats': [ 'markdown' ],
    }
  })

  #English: Write results to JSON instead of just printing to console
      #This result needs to be stored for the downstream RAG (Vector DB) flow to use.
  output_filename = "firecrawl_extracted_data.json"
      
  with open(output_filename, "w", encoding="utf-8") as f:
          json.dump(crawl_result, f, ensure_ascii=False, indent=4)
          
  logging.info(f"Firecrawl extraction completed. Data successfully saved to {output_filename}")

2026-03-19 01:15:41,231 - ERROR - FIRECRAWL_API_KEY is not set. Please set it in your environment or .env file.
